# 07 — Visuals and GitHub Export Assets

This notebook creates GitHub-ready charts and sample output tables from the modeling, FP-Growth, and segmentation outputs. The goal is to convert technical results into visuals that clearly communicate model performance, recommendation behavior, and business insights.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/instacart"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"
images_path = f"{base_path}/github_images"

dbutils.fs.mkdirs(images_path)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

In [0]:
def save_fig(filename):
    path = f"{images_path}/{filename}"
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved to: {path}")

## Visual 1 — Model comparison grouped chart

In [0]:
model_results = spark.read.parquet(f"{gold_path}/model_results")
model_pd = model_results.toPandas()

metrics = ["roc_auc", "pr_auc", "precision_at_10", "recall_at_10"]
metric_labels = ["ROC-AUC", "PR-AUC", "Precision@10", "Recall@10"]

x = np.arange(len(metrics))
width = 0.35

lr_values = model_pd[model_pd["model"] == "Logistic Regression"][metrics].values[0]
gbt_values = model_pd[model_pd["model"] == "GBTClassifier"][metrics].values[0]

plt.figure(figsize=(10, 5))
bars1 = plt.bar(x - width/2, lr_values, width, label="Logistic Regression")
bars2 = plt.bar(x + width/2, gbt_values, width, label="GBTClassifier")

plt.xticks(x, metric_labels)
plt.ylabel("Score")
plt.title("Model Comparison Across Evaluation Metrics")
plt.legend()
plt.ylim(0, 1)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            height + 0.015,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

save_fig("model_comparison_grouped.png")

## Visual 2 — Model lift table

In [0]:
lr = model_pd[model_pd["model"] == "Logistic Regression"].iloc[0]
gbt = model_pd[model_pd["model"] == "GBTClassifier"].iloc[0]

improvement = pd.DataFrame({
    "metric": metrics,
    "logistic_regression": [lr[m] for m in metrics],
    "gbt_classifier": [gbt[m] for m in metrics],
})

improvement["absolute_lift"] = improvement["gbt_classifier"] - improvement["logistic_regression"]
improvement["relative_lift_pct"] = improvement["absolute_lift"] / improvement["logistic_regression"] * 100

display(improvement)

## Visual 3 — Precision@K / Recall@K

In [0]:
k_results = pd.DataFrame({
    "k": [5, 10, 15, 20],
    "lr_precision": [0.3689875041695568, 0.2944763652003297, 0.25238110511184386, 0.22453571185620527],
    "lr_recall": [0.3899339580764242, 0.5641248108070833, 0.6738671783214241, 0.7490220145117609],
    "gbt_precision": [0.3837053857799022, 0.30572646142079, 0.2603096710421035, 0.2300529930508785],
    "gbt_recall": [0.405096030187751, 0.5806475678780788, 0.6875298822579208, 0.7603560294139683]
})

In [0]:
plt.figure(figsize=(9, 5))
plt.plot(k_results["k"], k_results["lr_precision"], marker="o", label="LR Precision@K")
plt.plot(k_results["k"], k_results["gbt_precision"], marker="o", label="GBT Precision@K")
plt.plot(k_results["k"], k_results["lr_recall"], marker="o", label="LR Recall@K")
plt.plot(k_results["k"], k_results["gbt_recall"], marker="o", label="GBT Recall@K")

plt.title("Recommendation Quality Tradeoff: Precision@K vs Recall@K")
plt.xlabel("Number of Recommended Products (K)")
plt.ylabel("Score")
plt.xticks(k_results["k"])
plt.legend()
plt.grid(True, alpha=0.3)

save_fig("precision_recall_at_k.png")

## Visual 4 — Feature importance

In [0]:
feature_importance = spark.read.parquet(f"{gold_path}/feature_importance_gbt")
feature_pd = feature_importance.orderBy(F.desc("importance")).limit(10).toPandas()

plt.figure(figsize=(10, 5))
plt.barh(feature_pd["feature"][::-1], feature_pd["importance"][::-1])
plt.title("Top Drivers of Reorder Prediction")
plt.xlabel("GBT Feature Importance")
plt.ylabel("Feature")

save_fig("feature_importance_gbt_top10.png")

## Visual 5 — Basket size distribution

In [0]:
basket_transactions = spark.read.parquet(f"{gold_path}/basket_transactions")

basket_pd = (
    basket_transactions
    .select("basket_size")
    .sample(False, 0.10, seed=42)
    .toPandas()
)

mean_basket = basket_pd["basket_size"].mean()
median_basket = basket_pd["basket_size"].median()

plt.figure(figsize=(9, 5))
plt.hist(basket_pd["basket_size"], bins=40)
plt.axvline(mean_basket, linestyle="--", label=f"Mean: {mean_basket:.1f}")
plt.axvline(median_basket, linestyle=":", label=f"Median: {median_basket:.1f}")

plt.title("Basket Size Distribution")
plt.xlabel("Number of Unique Products in Basket")
plt.ylabel("Number of Orders")
plt.legend()

save_fig("basket_size_distribution.png")

## Visual 6 — Segment distribution with real names

In [0]:
segment_names = {
    0: "Low-Frequency Small Basket Shoppers",
    1: "Snack & Beverage Routine Buyers",
    2: "Fresh Produce Loyalists",
    3: "Bulk Variety Shoppers",
    4: "High-Engagement Routine Reorderers"
}

segmented_users = spark.read.parquet(f"{gold_path}/segmented_users")

segment_dist_pd = (
    segmented_users
    .groupBy("cluster")
    .agg(F.count("*").alias("num_users"))
    .orderBy("cluster")
    .toPandas()
)

segment_dist_pd["segment_name"] = segment_dist_pd["cluster"].map(segment_names)

plt.figure(figsize=(10, 5))
plt.barh(segment_dist_pd["segment_name"][::-1], segment_dist_pd["num_users"][::-1])
plt.title("Customer Segment Distribution")
plt.xlabel("Number of Users")
plt.ylabel("Segment")

for i, value in enumerate(segment_dist_pd["num_users"][::-1]):
    plt.text(value, i, f" {value:,.0f}", va="center", fontsize=9)

save_fig("segment_distribution_named.png")

## Visual 7 — Segment profile heatmap with names

In [0]:
segment_profile = spark.read.parquet(f"{gold_path}/segment_profile")
profile_pd = segment_profile.toPandas()
profile_pd["segment_name"] = profile_pd["cluster"].map(segment_names)

heatmap_cols = [
    "avg_total_orders",
    "avg_reorder_rate",
    "avg_basket_size",
    "avg_days_between_orders",
    "avg_produce_share",
    "avg_dairy_share",
    "avg_snacks_share",
    "avg_beverages_share"
]

heatmap_labels = [
    "Orders",
    "Reorder Rate",
    "Basket Size",
    "Days Between Orders",
    "Produce Share",
    "Dairy Share",
    "Snacks Share",
    "Beverages Share"
]

heatmap_data = profile_pd[heatmap_cols]
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(11, 5))
plt.imshow(heatmap_norm, aspect="auto")

plt.xticks(range(len(heatmap_labels)), heatmap_labels, rotation=35, ha="right")
plt.yticks(range(len(profile_pd)), profile_pd["segment_name"])
plt.colorbar(label="Normalized Segment Score")

plt.title("Customer Segment Behavioral Profile")
plt.xlabel("Behavioral Feature")
plt.ylabel("Customer Segment")

save_fig("segment_profile_heatmap_named.png")

## Visual 8 — Top departments by segment

In [0]:
segment_top_departments = spark.read.parquet(f"{gold_path}/segment_top_departments")

top_dept_pd = (
    segment_top_departments
    .filter(F.col("rank") <= 3)
    .toPandas()
)

top_dept_pd["segment_name"] = top_dept_pd["cluster"].map(segment_names)

display(top_dept_pd[
    ["segment_name", "department", "department_share", "rank"]
].sort_values(["segment_name", "rank"]))

## Visual 9 — Cross-sell sample table

In [0]:
cross_sell = spark.read.parquet(f"{gold_path}/cross_sell_recommendations")

cross_sell_sample = (
    cross_sell
    .select(
        "basket_contains",
        "recommend_product",
        "recommend_department",
        "confidence",
        "lift"
    )
    .orderBy(F.desc("lift"), F.desc("confidence"))
    .limit(20)
)

display(cross_sell_sample)

cross_sell_sample.write.mode("overwrite").option("header", True).csv(
    f"{gold_path}/sample_cross_sell_recommendations_csv"
)

The cross-sell table shows interpretable basket rules from FP-Growth. These rules are useful for checkout add-ons and bundle recommendations, while the supervised model handles personalized reorder prediction.

## Visual 10 — Recommendation example table

In [0]:
top_recs = spark.read.parquet(f"{gold_path}/top10_recommendations_named")

best_example_user = (
    top_recs
    .groupBy("user_id")
    .agg(F.sum("label").alias("hits_in_top10"))
    .orderBy(F.desc("hits_in_top10"))
    .limit(1)
    .collect()[0]["user_id"]
)

sample_recs = (
    top_recs
    .filter(F.col("user_id") == best_example_user)
    .orderBy("rank")
    .select(
        "user_id",
        "rank",
        "product_name",
        "aisle",
        "department",
        F.round("score", 3).alias("score"),
        "label"
    )
)

display(sample_recs)

sample_recs.write.mode("overwrite").option("header", True).csv(
    f"{gold_path}/sample_top_recommendations_csv"
)